<a href="https://colab.research.google.com/github/zohrehasadi00/automatic_image_analysis/blob/main/notebooks/Modern_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Fundamental setup

In [ ]:
# import collection

import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from scipy.sparse import find

In [ ]:
#set seed for reproducibility
random_state = 42
torch.manual_seed(random_state)
np.random.seed(random_state)


In [ ]:
# connect Drive for data
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#path to image data
data_raw =[
    '/content/drive/MyDrive/AIA_SkinLesion_Projekt/data/raw/HAM10000_images_part_1',
    '/content/drive/MyDrive/AIA_SkinLesion_Projekt/data/raw/HAM10000_images_part_2'
    ]

# load metadata
csv_path = '/content/drive/MyDrive/AIA_SkinLesion_Projekt/HAM10000_metadata_converted.csv'
df_metadata = pd.read_csv(csv_path)

#load splits
csv_train = '/content/drive/MyDrive/AIA_SkinLesion_Projekt/splits/train.csv'
df_train = pd.read_csv(csv_train)

csv_test = '/content/drive/MyDrive/AIA_SkinLesion_Projekt/splits/test.csv'
df_test = pd.read_csv(csv_test)

csv_val = '/content/drive/MyDrive/AIA_SkinLesion_Projekt/splits/val.csv'
df_val = pd.read_csv(csv_val)

# Data preprocessing for modern pipeline

In [ ]:
# EfficientNet-B0 will 224x224 andere resolution sind auch möglich und ImageNet-Normalisierung

# define image tranformations for train and test/validation set
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # avoid overfitting in training data
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## function for searching images from drive
def find_image_path(img_id, directories):
    img_name = img_id + '.jpg'
    for directory in directories:
        path = os.path.join(directory, img_name)
        if os.path.exists(path):
            return path
    return None

# Dataset class
class SkinDataset(Dataset):
    def __init__(self, dataframe, img_dirs, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dirs = img_dirs
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.loc[idx, 'image_id']
        img_name = img_id + '.jpg'
        img_path = None

        # search in both folders (part 1&2) for the image
        img_path = find_image_path(img_id, self.img_dirs)

        if img_path is None:
            raise FileNotFoundError(f"Bild {img_name} nicht gefunden!")

        image = Image.open(img_path).convert('RGB')
        label = self.data.loc[idx, 'label']

        if self.transform:
            image = self.transform(image)

        # PyTorch expects labels as Float for binary loss (BCEWithLogitsLoss)
        return image, torch.tensor(label, dtype=torch.float32), img_id



# create data sets
train_dataset = SkinDataset(train_df, data_raw, transform=train_transforms)
val_dataset = SkinDataset(val_df, data_raw, transform=test_transforms)
test_dataset = SkinDataset(test_df, data_raw, transform=test_transforms)

# create data loader
b_size = 32
train_loader = DataLoader(train_dataset, batch_size=b_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=b_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=b_size, shuffle=False)

In [ ]:
### Example output
images, labels, image_ids = next(iter(train_loader))


tensor_img = images[-1]
label = labels[-1]
last_id = image_ids[-1]  # Das ist z.B. "ISIC_0024306"

print(f"Loading original image with id: {last_id}.jpg ...")

# image after transformation
img_np = tensor_img.numpy().transpose((1, 2, 0))
#mean = np.array([0.485, 0.456, 0.406])
#std = np.array([0.229, 0.224, 0.225])
#bild_np = std * bild_np + mean
minimum = img_np.min()
maximum = img_np.max()
img_np = (img_np - minimum) / (maximum - minimum)

# original image
img_name = last_id + '.jpg'
original_path = find_image_path(last_id, data_raw)

# show images
if original_path:
    original_image = Image.open(original_path).convert('RGB')
    label_name = "malignant (1)" if label.item() == 1.0 else "benign (0)"

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"Control check: last image in batch\nid: {last_id} | diagnosis: {label_name}", fontsize=14, fontweight='bold')

    axes[0].imshow(original_image)
    axes[0].set_title(f"before \nfile name: {img_name}\nsize: {original_image.size}", fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(img_np)
    axes[1].set_title(f"after \nsize: {tensor_img.shape[1]}x{tensor_img.shape[2]}", fontsize=12)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print(f"Error: {img_name} could not be found!")

check_row = df_metadata[df_metadata['image_id'] == last_id]

print(f"Searched for id: {last_id}")
print(f"ID from csv:\n{check_row[['image_id', 'label']]}")

# additional check
csv_label = check_row['label'].values[0]
batch_label = int(label.item())

if csv_label == batch_label:
    print("\n Matches csv")
else:
    print("\n Differs from csv")

print("finished")